In [34]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from scipy.special import factorial

In [35]:
def erlang_b(A, m):
    """Calculates the theoretical blocking probability using Erlang's B-formula."""
    numerator = (A**m) / factorial(m)
    denominator = sum([(A**i) / factorial(i) for i in range(m + 1)])
    return numerator / denominator

def mean_confidence_interval(data, confidence=0.95):
    """Calculates the mean and confidence interval for the blocked fraction."""
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), stats.sem(a)
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h


In [36]:
import heapq

def simulate_blocking_system(m, num_customers, arrival_gen, service_gen):
    """
    Simulates a system with m servers and no waiting room.
    Returns the fraction of blocked customers.
    """
    clock = 0.0
    blocked_count = 0
    # Priority queue stores departure times of customers currently in service
    servers = [] 
    
    for _ in range(num_customers):
        # Time of the current arrival
        clock += arrival_gen()
        
        # Remove customers who have finished service by the time this customer arrives
        while servers and servers[0] <= clock:
            heapq.heappop(servers)
            
        if len(servers) < m:
            # Assign a server: generate service time and schedule departure
            service_time = service_gen()
            heapq.heappush(servers, clock + service_time)
        else:
            # All m servers are busy; customer is blocked
            blocked_count += 1
            
    return blocked_count / num_customers

In [37]:
np.random.seed(42)  # For reproducibility
# Parameters
m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000
num_runs = 10

# Distribution Generators
arrival_gen = lambda: np.random.exponential(mean_interarrival)
service_gen = lambda: np.random.exponential(mean_service)

# Execution
results_p1 = [simulate_blocking_system(m, num_customers, arrival_gen, service_gen) for _ in range(num_runs)]
mean_p1, ci_low_p1, ci_high_p1 = mean_confidence_interval(results_p1)
exact_p1 = erlang_b(8.0, 10)

print(f"Part 1 - Poisson Arrivals / Exponential Service:")
print(f"  Observed Blocked Fraction: {mean_p1:.4f} (95% CI: [{ci_low_p1:.4f}, {ci_high_p1:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")

Part 1 - Poisson Arrivals / Exponential Service:
  Observed Blocked Fraction: 0.1185 (95% CI: [0.1142, 0.1228])
  Exact Erlang B Solution:   0.1217


In [38]:
np.random.seed(42)  # For reproducibility

# 2a: Erlang distributed inter-arrival times (mean 1)
# Note: For an Erlang(k) with mean 1, the scale is 1/k.
k_erlang = 2 # Example: Erlang-2
erlang_arrival = lambda: np.random.gamma(k_erlang, 1.0/k_erlang)

# 2b: Hyperexponential inter-arrival times
# p1=0.8, lambda1=0.8333, p2=0.2, lambda2=5.0
def hyperexponential_arrival():
    if np.random.rand() < 0.8:
        return np.random.exponential(1/0.8333)
    else:
        return np.random.exponential(1/5.0)
results_p2a = [simulate_blocking_system(m, num_customers, erlang_arrival, service_gen) for _ in range(num_runs)]
mean_p2a, ci_low_p2a, ci_high_p2a = mean_confidence_interval(results_p2a)

# Running 2b
results_p2b = [simulate_blocking_system(m, num_customers, hyperexponential_arrival, service_gen) for _ in range(num_runs)]
mean_p2b, ci_low_p2b, ci_high_p2b = mean_confidence_interval(results_p2b)

print(f"\nPart 2a - Erlang Arrivals:")
print(f"  Observed Blocked Fraction: {mean_p2a:.4f} (95% CI: [{ci_low_p2a:.4f}, {ci_high_p2a:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")

print(f"\nPart 2b - Hyperexponential Arrivals:")
print(f"  Observed Blocked Fraction: {mean_p2b:.4f} (95% CI: [{ci_low_p2b:.4f}, {ci_high_p2b:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")


Part 2a - Erlang Arrivals:
  Observed Blocked Fraction: 0.0946 (95% CI: [0.0924, 0.0969])
  Exact Erlang B Solution:   0.1217

Part 2b - Hyperexponential Arrivals:
  Observed Blocked Fraction: 0.1397 (95% CI: [0.1341, 0.1453])
  Exact Erlang B Solution:   0.1217


In [41]:
np.random.seed(42)  # For reproducibility

# 3a: Constant service time
constant_service = lambda: 8.0

# 3b: Pareto distributed service times (mean 8, beta=1 is often used in exercises)
# The mean of a Pareto distribution is (k * beta) / (k - 1)
def pareto_service(k, mean_target=8.0):
    # Solve for beta: beta = mean * (k - 1) / k
    beta = mean_target * (k - 1) / k
    return (np.random.pareto(k) + 1) * beta

results_p3a = [simulate_blocking_system(m, num_customers, arrival_gen, constant_service) for _ in range(num_runs)]
mean_p3a, ci_low_p3a, ci_high_p3a = mean_confidence_interval(results_p3a)

# Run for Pareto k=2.05
pareto_gen = lambda: pareto_service(1.05)
pareto_gen = lambda: pareto_service(2.05)

results_p3b_105 = [simulate_blocking_system(m, num_customers, arrival_gen, pareto_gen) for _ in range(num_runs)]
mean_p3b_105, ci_low_p3b_105, ci_high_p3b_105 = mean_confidence_interval(results_p3b_105)
results_p3b_205 = [simulate_blocking_system(m, num_customers, arrival_gen, pareto_gen) for _ in range(num_runs)]
mean_p3b_205, ci_low_p3b_205, ci_high_p3b_205 = mean_confidence_interval(results_p3b_205)

print(f"\nPart 3a - Constant Service:")
print(f"  Observed Blocked Fraction: {mean_p3a:.4f} (95% CI: [{ci_low_p3a:.4f}, {ci_high_p3a:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")

print(f"\nPart 3b - Pareto Service (k=1.05):")
print(f"  Observed Blocked Fraction: {mean_p3b_105:.4f} (95% CI: [{ci_low_p3b_105:.4f}, {ci_high_p3b_105:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")
print(f"\nPart 3b - Pareto Service (k=2.05):")
print(f"  Observed Blocked Fraction: {mean_p3b_205:.4f} (95% CI: [{ci_low_p3b_205:.4f}, {ci_high_p3b_205:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")



Part 3a - Constant Service:
  Observed Blocked Fraction: 0.1225 (95% CI: [0.1183, 0.1268])
  Exact Erlang B Solution:   0.1217

Part 3b - Pareto Service (k=1.05):
  Observed Blocked Fraction: 0.1205 (95% CI: [0.1148, 0.1262])
  Exact Erlang B Solution:   0.1217

Part 3b - Pareto Service (k=2.05):
  Observed Blocked Fraction: 0.1184 (95% CI: [0.1148, 0.1220])
  Exact Erlang B Solution:   0.1217
